# **EDA — Exploratory Data Analysis and Hypothesis Testing**

## Objectives

* Profile the cleaned dataset and describe the campaign, the client base and the outcome distribution
* Test five stated hypotheses using appropriate statistical methods
* Report effect sizes alongside p-values, since a sample of this size will return significance for trivially small differences
* Test whether call duration is usable as a predictor, or whether it constitutes target leakage
* Save all test results for use in the dashboard

## Inputs

* Data_Set/clean_data/v1/bank_marketing_cleaned.csv — the cleaned dataset produced by Notebook 01

## Outputs

* Data_Set/outputs/v1/hypothesis_results.csv — test statistic, p-value, effect size and outcome for each hypothesis
* Data_Set/outputs/v1/descriptive_summary.csv — summary statistics for the cleaned dataset

## Additional Comments

* With 41,176 records, statistical significance is almost guaranteed for any real
  difference. Every test therefore reports an effect size (Cramér's V for categorical
  associations, rank-biserial correlation for Mann-Whitney tests) so that the practical
  size of each difference can be judged separately from its statistical significance.

* H2 and H5 are framed as null hypotheses: the analysis tests for the absence of
  disparity across education level and the absence of a usable relationship for duration.
  A "not supported" result is a finding about the campaign, not a failure of the analysis.

* Findings describe association only. The data is observational, and no causal claim is
  made about why any group subscribed at a higher or lower rate.



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/Users/apple/Desktop/fair-marketing-analytics/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'/Users/apple/Desktop/fair-marketing-analytics'

# Section 1 — Load and Profile

In [4]:
import pandas as pd
import numpy as np
import os
from scipy import stats

pd.set_option('display.max_columns', None)

version = 'v1'
clean_dir = f'Data_Set/clean_data/{version}'
output_dir = f'Data_Set/outputs/{version}'
os.makedirs(output_dir, exist_ok=True)

print(f"Reading from: {clean_dir}")
print(f"Writing to:   {output_dir}")

Reading from: Data_Set/clean_data/v1
Writing to:   Data_Set/outputs/v1


In [5]:
df = pd.read_csv(f'{clean_dir}/bank_marketing_cleaned.csv')

print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns\n")
print(df.dtypes)

Loaded 41,176 rows and 23 columns

age                    int64
job                   object
marital               object
education             object
housing               object
loan                  object
contact               object
month                 object
day_of_week           object
duration               int64
campaign               int64
pdays                float64
previous               int64
poutcome              object
emp_var_rate         float64
cons_price_idx       float64
cons_conf_idx        float64
euribor3m            float64
nr_employed          float64
default_disclosed      int64
contacted_before       int64
subscribed             int64
age_band              object
dtype: object


## Outcome distribution

The target variable is the proportion of contacted clients who subscribed to a term
deposit. This sets the baseline against which every subgroup difference is judged.

## Numeric variables

In [6]:
target = df['subscribed'].value_counts()
target_pct = df['subscribed'].value_counts(normalize=True)

print(f"Did not subscribe: {target[0]:,} ({target_pct[0]:.2%})")
print(f"Subscribed:        {target[1]:,} ({target_pct[1]:.2%})")
print(f"\nClass ratio: {target[0] / target[1]:.1f} to 1")

Did not subscribe: 36,537 (88.73%)
Subscribed:        4,639 (11.27%)

Class ratio: 7.9 to 1


In [7]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
desc = df[numeric_cols].describe().T.round(2)
desc['missing'] = df[numeric_cols].isnull().sum()

desc.to_csv(f'{output_dir}/descriptive_summary.csv')
desc

,count,mean,std,min,25%,50%,75%,max,missing
age,41176.0,40.02,10.42,17.00,32.00,38.00,47.00,98.00,0
duration,41176.0,258.32,259.31,0.00,102.00,180.00,319.00,4918.00,0
campaign,41176.0,2.57,2.77,1.00,1.00,2.00,3.00,56.00,0
pdays,1515.0,6.01,3.82,0.00,3.00,6.00,7.00,27.00,39661
previous,41176.0,0.17,0.49,0.00,0.00,0.00,0.00,7.00,0
emp_var_rate,41176.0,0.08,1.57,-3.40,-1.80,1.10,1.40,1.40,0
cons_price_idx,41176.0,93.58,0.58,92.20,93.08,93.75,93.99,94.77,0
cons_conf_idx,41176.0,-40.50,4.63,-50.80,-42.70,-41.80,-36.40,-26.90,0
euribor3m,41176.0,3.62,1.73,0.63,1.34,4.86,4.96,5.04,0
nr_employed,41176.0,5167.03,72.25,4963.60,5099.10,5191.00,5228.10,5228.10,0


## Categorical variables

In [8]:
cat_cols = df.select_dtypes(include='object').columns.tolist()

for col in cat_cols:
    print(f"\n{col} — {df[col].nunique()} categories, {df[col].isnull().sum()} missing")
    print(df[col].value_counts(dropna=False).head(12).to_string())


job — 11 categories, 330 missing
job
admin.           10419
blue-collar       9253
technician        6739
services          3967
management        2924
retired           1718
entrepreneur      1456
self-employed     1421
housemaid         1060
unemployed        1014
student            875
NaN                330

marital — 3 categories, 80 missing
marital
married     24921
single      11564
divorced     4611
NaN            80

education — 7 categories, 1730 missing
education
university.degree      12164
high.school             9512
basic.9y                6045
professional.course     5240
basic.4y                4176
basic.6y                2291
NaN                     1730
illiterate                18

housing — 2 categories, 990 missing
housing
yes    21571
no     18615
NaN      990

loan — 2 categories, 990 missing
loan
no     33938
yes     6248
NaN      990

contact — 2 categories, 0 missing
contact
cellular     26135
telephone    15041

month — 10 categories, 0 missing
month
may  

## Subscription rate by group

The overall subscription rate is 11.3%. This section reports the rate within each
category of the demographic and financial fields, which is the basis for the hypothesis
tests in Section 2.

In [9]:
def rate_by(col):
    out = df.groupby(col, observed=True, dropna=False).agg(
        contacted=('subscribed', 'size'),
        subscribed=('subscribed', 'sum')
    )
    out['rate'] = (out['subscribed'] / out['contacted'] * 100).round(2)
    return out.sort_values('rate', ascending=False)

for col in ['age_band', 'job', 'education', 'marital', 'housing', 'loan']:
    print(f"\n=== {col} ===")
    print(rate_by(col).to_string())


=== age_band ===
             contacted  subscribed   rate
age_band                                 
60 and over       1192         472  39.60
Under 25          1067         256  23.99
25-59            38917        3911  10.05

=== job ===
               contacted  subscribed   rate
job                                        
student              875         275  31.43
retired             1718         434  25.26
unemployed          1014         144  14.20
admin.             10419        1351  12.97
management          2924         328  11.22
NaN                  330          37  11.21
technician          6739         730  10.83
self-employed       1421         149  10.49
housemaid           1060         106  10.00
entrepreneur        1456         124   8.52
services            3967         323   8.14
blue-collar         9253         638   6.90

=== education ===
                     contacted  subscribed   rate
education                                        
illiterate              

### Findings

---

# Section 2

Section 2 content

---

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)
